# Wind Farm Yaw Control via PPO (Proximal Policy Optimization)

This notebook trains a PPO agent to maximize total power output by controlling the yaw angles of turbines, sweeping across **all 19 wfcrl environments** (Turb_TCRWP, Turb1/2/3/6/16/32_RowN, Ablaincourt, HornsRev1/2, Ormonde, WMR - both Floris and Fastfarm variants where available).

After the sweep, pick any trained env via `SELECTED_ENV` to drill into per-env visualizations, or scroll to the bottom for cross-environment summary plots.

**Baselines to beat** (Scenario 2 - random wind conditions, `Turb3_Row1_Floris`):
- Note: Sc.1 fixed-wind baselines (228/232) do not apply here; Sc.2 rewards vary with sampled wind conditions.
- WFCRL ([arXiv:2501.13592](https://arxiv.org/abs/2501.13592)), Table 7 - `Turb3_Row1_Floris` FLORIS, 200k steps:
  - IPPO: 406.0 +- 10.5
  - MAPPO: 372.4 +- 15.5


## Imports

In [24]:
# If you wanna run on Colab:
!pip install -q git+https://github.com/ifpen/wfcrl-env.git tabpfn-client torch-geometric python-dotenv "numpy==2.0.2" "scipy==1.16.3" "scikit-learn==1.6.1"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done


In [25]:
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

from wfcrl import environments as envs

sns.set_theme(style="whitegrid")
SEED = 13
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, module="floris")


In [26]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


## Environment Setup

In [27]:
# All 19 wfcrl environments. Fastfarm variants need an external FAST.Farm
# binary; if not installed, those iterations will fail and be recorded as
# errors in `results` rather than killing the whole sweep.
ENV_IDS = [
    # "Turb_TCRWP_Fastfarm",  
    "Turb_TCRWP_Floris",
    # "Turb1_Row1_Fastfarm",  
    "Turb1_Row1_Floris",
    # "Turb2_Row1_Fastfarm",  
    "Turb2_Row1_Floris",
    # "Turb3_Row1_Fastfarm",  
    "Turb3_Row1_Floris",
    # "Turb6_Row2_Fastfarm",  
    "Turb6_Row2_Floris",
    # "Turb16_Row5_Fastfarm", 
    "Turb16_Row5_Floris",
    # "Turb32_Row5_Fastfarm", 
    "Turb32_Row5_Floris",
    "Ablaincourt_Floris",
    "HornsRev1_Floris",     "HornsRev2_Floris",
    "Ormonde_Floris",
    "WMR_Floris",
]
print(f"Will sweep over {len(ENV_IDS)} env_ids")


Will sweep over 12 env_ids


In [28]:
# Observation helpers. The sweep loop sets `_OBS_LOW`, `_OBS_HIGH`, `N_TURBINES`,
# `OBS_DIM`, `ACT_DIM`, and `env` per env_id; these helpers read whatever globals
# are currently in scope.
_OBS_KEYS = ["yaw", "freewind_measurements", "wind_speed", "wind_direction"]
MAX_STEPS = 150  # max steps per episode (matches env max_num_steps)


def flatten_obs(obs: dict, step: int = 0, max_steps: int = MAX_STEPS) -> np.ndarray:
    """Convert Dict obs to a normalized flat vector in [-1, 1], appending normalized timestep.

    Raw features span very different scales (yaw: +-45, wind direction: 0-360).
    Normalizing to [-1, 1] prevents wind-direction values from dominating the
    first linear layer and keeps all gradients on the same scale.
    The timestep t/T in [0, 1] lets the policy distinguish early exploration
    from late-episode convergence.
    """
    raw = np.concatenate([obs[k] for k in _OBS_KEYS]).astype(np.float32)
    normalized = 2.0 * (raw - _OBS_LOW) / (_OBS_HIGH - _OBS_LOW) - 1.0
    t_norm = np.array([step / max_steps], dtype=np.float32)
    # Guard against NaN/inf from the simulator (e.g. async Floris I/O races).
    return np.nan_to_num(np.concatenate([normalized, t_norm]), nan=0.0, posinf=1.0, neginf=-1.0)


def flatten_obs_batch(obs_batch: dict, steps: np.ndarray = None, max_steps: int = MAX_STEPS) -> np.ndarray:
    """Vectorized version of flatten_obs. obs_batch[key] has shape (N, ...). Returns (N, OBS_DIM)."""
    raw = np.concatenate([obs_batch[k] for k in _OBS_KEYS], axis=-1).astype(np.float32)
    normalized = 2.0 * (raw - _OBS_LOW) / (_OBS_HIGH - _OBS_LOW) - 1.0
    N = normalized.shape[0]
    if steps is None:
        steps = np.zeros(N, dtype=np.float32)
    t_norms = (steps / max_steps).reshape(-1, 1).astype(np.float32)
    return np.nan_to_num(np.concatenate([normalized, t_norms], axis=-1), nan=0.0, posinf=1.0, neginf=-1.0)


def env_reset(seed_val, options):
    """Call env.reset() and always return just the obs dict. Reads global `env`."""
    result = env.reset(seed=seed_val, options=options)
    return result[0] if isinstance(result, tuple) else result


## Print helper

In [38]:
# ---- Progress-print throttling -------------------------------------
STATUS_PRINT_EVERY_SEC = 30.0
_last_status_print = 0.0

def print_status(msg, *, force=False):
    global _last_status_print
    now = time.time()
    if force or (now - _last_status_print) > STATUS_PRINT_EVERY_SEC:
        print("\r\033[K" + msg, end="", flush=True)  # erase current line, then print
        _last_status_print = now

## Actor-Critic Network

In [39]:
class ActorCritic(nn.Module):
    """
    Split Actor-Critic for continuous actions.
    Separate backbones prevent VF loss gradients from corrupting actor weights.
    Actor: outputs a Gaussian (mean, log_std) over actions.
    Critic: outputs a scalar state value.
    """
    def __init__(self, obs_dim: int, act_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.actor_backbone = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
        )
        self.critic_backbone = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
        )
        self.actor_mean    = nn.Linear(hidden_dim, act_dim)
        self.actor_log_std = nn.Parameter(torch.zeros(act_dim))  # learnable log-std
        self.critic        = nn.Linear(hidden_dim, 1)

        # Orthogonal init (standard for PPO)
        for backbone in (self.actor_backbone, self.critic_backbone):
            for layer in backbone:
                if isinstance(layer, nn.Linear):
                    nn.init.orthogonal_(layer.weight, gain=np.sqrt(2))
                    nn.init.zeros_(layer.bias)
        nn.init.orthogonal_(self.actor_mean.weight, gain=0.01)
        nn.init.zeros_(self.actor_mean.bias)
        nn.init.orthogonal_(self.critic.weight, gain=1.0)
        nn.init.zeros_(self.critic.bias)

    def _dist(self, obs: torch.Tensor) -> Normal:
        features = self.actor_backbone(obs)
        mean = self.actor_mean(features)
        std  = self.actor_log_std.clamp(-4.0, 0.5).exp().expand_as(mean)
        return Normal(mean, std)

    def act(self, obs: torch.Tensor):
        """Sample action; return (action, log_prob, value)."""
        actor_features  = self.actor_backbone(obs)
        critic_features = self.critic_backbone(obs)
        std  = self.actor_log_std.clamp(-4.0, 0.5).exp()
        dist = Normal(self.actor_mean(actor_features), std)
        action   = dist.sample()
        log_prob = dist.log_prob(action).sum(-1)
        value    = self.critic(critic_features).squeeze(-1)
        return action, log_prob, value

    def evaluate(self, obs: torch.Tensor, action: torch.Tensor):
        """Re-evaluate stored (obs, action) pairs for the PPO loss."""
        actor_features  = self.actor_backbone(obs)
        critic_features = self.critic_backbone(obs)
        std  = self.actor_log_std.clamp(-4.0, 0.5).exp()
        dist = Normal(self.actor_mean(actor_features), std)
        log_prob = dist.log_prob(action).sum(-1)
        entropy  = dist.entropy().sum(-1)
        value    = self.critic(critic_features).squeeze(-1)
        return log_prob, entropy, value

## PPO Hyperparameters

In [40]:
# -- Training budget -----------------------------------------------------------
ROLLOUTS_PER_UPDATE = 16  # episodes collected before each ppo_update call
N_EPISODES = 4          # update cycles (200 x 16 x 150 ~= 480K steps)
EVAL_EVERY = 20           # print every N update cycles

# -- PPO update ----------------------------------------------------------------
LR              = 3e-4
MIN_LR          = 1e-5
N_EPOCHS        = 20
BATCH_SIZE      = 32
GAMMA           = 0.99
GAE_LAMBDA      = 0.95
CLIP_EPS        = 0.2
VF_COEF         = 0.5
ENT_COEF_START  = 0.01    # high entropy early -> exploration
ENT_COEF_END    = 0.001   # decay to low entropy late -> exploitation
ENT_COEF        = ENT_COEF_START  # alias for ppo_update default arg
MAX_GRAD        = 0.5

# Differential rewards are ~+-0.1-0.5 MW/step; scale so cumulative return is O(1)
REWARD_SCALE    = 20.0

# Terminal bonus: extra weight on the final-step differential reward.
# Encourages the agent to find the optimal yaw and be there at episode end.
TERMINAL_SCALE  = 5.0

# Convergence penalty: penalize large yaw deltas quadratically late in the episode.
# Penalty = CONVERGENCE_COEF * (t/T)^2 * mean(|action|), so early moves are free.
CONVERGENCE_COEF = 0.05

# Scenario 2: None = sample random wind speed and direction each episode
WIND_OPTS = None

In [41]:
import sys, os, base64

def _find_module_dir(filename, start='.', max_levels=5):
    d = os.path.abspath(start)
    for _ in range(max_levels):
        if os.path.exists(os.path.join(d, filename)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None

_module_dir = (_find_module_dir('wfcrl_vec_utils.py', '.') or
               _find_module_dir('wfcrl_vec_utils.py', '..'))

if _module_dir is None:
    # Not found on disk (fresh Colab session with no repo clone).
    # Decode the embedded copy and write it to /tmp so that spawned worker
    # processes can import it by module path (required by multiprocessing spawn).
    _MODULE_B64 = (
        "IiIiDQpQYXJhbGxlbCBXRkNSTCBlbnZpcm9ubWVudHMgZm9yIFBQTyByb2xsb3V0IGNvbGxlY3Rp"
        "b24uDQoNCk11c3QgYmUgYSBzZXBhcmF0ZSBpbXBvcnRhYmxlIG1vZHVsZSAobm90IGRlZmluZWQg"
        "aW4gdGhlIG5vdGVib29rKSBzbyB0aGF0DQptdWx0aXByb2Nlc3NpbmcgJ3NwYXduJyBvbiBXaW5k"
        "b3dzIGNhbiBmaW5kIHRoZSB3b3JrZXIgZnVuY3Rpb24gYnkgbW9kdWxlIHBhdGguDQoiIiINCmlt"
        "cG9ydCBtdWx0aXByb2Nlc3NpbmcgYXMgbXANCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IG9z"
        "DQppbXBvcnQgcGlja2xlDQppbXBvcnQgc2h1dGlsDQppbXBvcnQgc29ja2V0DQppbXBvcnQgc3Vi"
        "cHJvY2Vzcw0KaW1wb3J0IHN5cw0KZnJvbSBtdWx0aXByb2Nlc3NpbmcuY29ubmVjdGlvbiBpbXBv"
        "cnQgTGlzdGVuZXINCg0KDQpkZWYgX2Vudl93b3JrZXIoc2VlZCwgcGlwZSwgZW52X2lkKToNCiAg"
        "ICAiIiJXb3JrZXIgcHJvY2VzczogY3JlYXRlcyBpdHMgb3duIGVudiBmcm9tIGEgc2VlZCwgc2Vy"
        "dmVzIGNvbW1hbmRzIG92ZXIgYSBQaXBlLg0KDQogICAgRGVmaW5lZCBhdCBtb2R1bGUgbGV2ZWwg"
        "c28gc3Bhd24gY2FuIGltcG9ydCBpdCBieSBwYXRoIC0gbm8gY2xvc3VyZSwgbm8gcGlja2xpbmcg"
        "aXNzdWVzLg0KICAgICIiIg0KICAgIGZyb20gd2ZjcmwgaW1wb3J0IGVudmlyb25tZW50cyBhcyBl"
        "bnZzICAjIGltcG9ydCBpbnNpZGUgd29ya2VyIGFmdGVyIHNwYXduDQoNCiAgICBlID0gZW52cy5t"
        "YWtlKA0KICAgICAgICBlbnZfaWQsDQogICAgICAgIG1heF9udW1fc3RlcHM9MTUwLA0KICAgICAg"
        "ICBjb250cm9scz17InlhdyI6ICgtNDUsIDQ1LCA1KX0sDQogICAgICAgIGNvbnRpbnVvdXNfY29u"
        "dHJvbD1UcnVlLA0KICAgICAgICBsb2c9RmFsc2UsDQogICAgKQ0KICAgIHBpcGUuc2VuZCgicmVh"
        "ZHkiKQ0KDQogICAgd2hpbGUgVHJ1ZToNCiAgICAgICAgY21kLCBkYXRhID0gcGlwZS5yZWN2KCkN"
        "Cg0KICAgICAgICBpZiBjbWQgPT0gInJlc2V0IjoNCiAgICAgICAgICAgIHJlc3VsdCA9IGUucmVz"
        "ZXQoKipkYXRhKQ0KICAgICAgICAgICAgb2JzID0gcmVzdWx0WzBdIGlmIGlzaW5zdGFuY2UocmVz"
        "dWx0LCB0dXBsZSkgZWxzZSByZXN1bHQNCiAgICAgICAgICAgIHBpcGUuc2VuZChvYnMpDQoNCiAg"
        "ICAgICAgZWxpZiBjbWQgPT0gInN0ZXAiOg0KICAgICAgICAgICAgb2JzLCByZXdhcmQsIHRlcm1p"
        "bmF0ZWQsIHRydW5jYXRlZCwgaW5mbyA9IGUuc3RlcChkYXRhKQ0KICAgICAgICAgICAgcGlwZS5z"
        "ZW5kKChvYnMsIGZsb2F0KG5wLnNxdWVlemUocmV3YXJkKSksIGJvb2wodGVybWluYXRlZCksIGJv"
        "b2wodHJ1bmNhdGVkKSwgaW5mbykpDQoNCiAgICAgICAgZWxpZiBjbWQgPT0gImNsb3NlIjoNCiAg"
        "ICAgICAgICAgIGUuY2xvc2UoKQ0KICAgICAgICAgICAgYnJlYWsNCg0KDQpjbGFzcyBQYXJhbGxl"
        "bFZlY0VudjoNCiAgICAiIiJQYXJhbGxlbCBXRkNSTCBlbnZpcm9ubWVudHMgdXNpbmcgbXVsdGlw"
        "cm9jZXNzaW5nLg0KDQogICAgV29ya3Mgb24gV2luZG93cyAoc3Bhd24pIGFuZCBMaW51eC9Db2xh"
        "YiAoc3Bhd24gZXhwbGljaXRseSwgb3IgZm9yayBpbXBsaWNpdGx5KS4NCiAgICBUaGUgQVBJIG1p"
        "cnJvcnMgZ3ltbmFzaXVtIFZlY3RvckVudjogcmVzZXQoKSBhbmQgc3RlcCgpIHJldHVybiBiYXRj"
        "aGVkIGFycmF5cy4NCiAgICAiIiINCg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBzZWVkcywgZW52"
        "X2lkKToNCiAgICAgICAgY3R4ID0gbXAuZ2V0X2NvbnRleHQoInNwYXduIikNCiAgICAgICAgc2Vs"
        "Zi5uID0gbGVuKHNlZWRzKQ0KICAgICAgICBzZWxmLmVudl9pZCA9IGVudl9pZA0KICAgICAgICBz"
        "ZWxmLnBpcGVzID0gW10NCiAgICAgICAgc2VsZi5wcm9jcyA9IFtdDQoNCiAgICAgICAgZm9yIHNl"
        "ZWQgaW4gc2VlZHM6DQogICAgICAgICAgICBwYXJlbnQsIGNoaWxkID0gY3R4LlBpcGUoKQ0KICAg"
        "ICAgICAgICAgcHJvYyA9IGN0eC5Qcm9jZXNzKHRhcmdldD1fZW52X3dvcmtlciwgYXJncz0oc2Vl"
        "ZCwgY2hpbGQsIGVudl9pZCksIGRhZW1vbj1UcnVlKQ0KICAgICAgICAgICAgcHJvYy5zdGFydCgp"
        "DQogICAgICAgICAgICBjaGlsZC5jbG9zZSgpICAjIHBhcmVudCBkb2Vzbid0IG5lZWQgdGhlIGNo"
        "aWxkIGVuZA0KICAgICAgICAgICAgYXNzZXJ0IHBhcmVudC5yZWN2KCkgPT0gInJlYWR5IiwgZiJX"
        "b3JrZXIgd2l0aCBzZWVkIHtzZWVkfSBmYWlsZWQgdG8gc3RhcnQiDQogICAgICAgICAgICBzZWxm"
        "LnBpcGVzLmFwcGVuZChwYXJlbnQpDQogICAgICAgICAgICBzZWxmLnByb2NzLmFwcGVuZChwcm9j"
        "KQ0KDQogICAgZGVmIHJlc2V0KHNlbGYsIHNlZWQ9Tm9uZSwgb3B0aW9ucz1Ob25lKToNCiAgICAg"
        "ICAgc2VlZHMgPSBzZWVkIGlmIGlzaW5zdGFuY2Uoc2VlZCwgbGlzdCkgZWxzZSBbc2VlZF0gKiBz"
        "ZWxmLm4NCiAgICAgICAgZm9yIGksIHBpcGUgaW4gZW51bWVyYXRlKHNlbGYucGlwZXMpOg0KICAg"
        "ICAgICAgICAgcGlwZS5zZW5kKCgicmVzZXQiLCB7InNlZWQiOiBzZWVkc1tpXSwgIm9wdGlvbnMi"
        "OiBvcHRpb25zfSkpDQogICAgICAgIG9ic19saXN0ID0gW3BpcGUucmVjdigpIGZvciBwaXBlIGlu"
        "IHNlbGYucGlwZXNdDQogICAgICAgIHJldHVybiB7azogbnAuc3RhY2soW29ba10gZm9yIG8gaW4g"
        "b2JzX2xpc3RdKSBmb3IgayBpbiBvYnNfbGlzdFswXX0sIHt9DQoNCiAgICBkZWYgc3RlcChzZWxm"
        "LCBhY3Rpb25fZGljdCk6DQogICAgICAgIGZvciBpLCBwaXBlIGluIGVudW1lcmF0ZShzZWxmLnBp"
        "cGVzKToNCiAgICAgICAgICAgIHBpcGUuc2VuZCgoInN0ZXAiLCB7azogdltpXSBmb3IgaywgdiBp"
        "biBhY3Rpb25fZGljdC5pdGVtcygpfSkpDQogICAgICAgIHJlc3VsdHMgPSBbcGlwZS5yZWN2KCkg"
        "Zm9yIHBpcGUgaW4gc2VsZi5waXBlc10NCiAgICAgICAgb2JzX2xpc3QgPSBbclswXSBmb3IgciBp"
        "biByZXN1bHRzXQ0KICAgICAgICByZXR1cm4gKA0KICAgICAgICAgICAge2s6IG5wLnN0YWNrKFtv"
        "W2tdIGZvciBvIGluIG9ic19saXN0XSkgZm9yIGsgaW4gb2JzX2xpc3RbMF19LA0KICAgICAgICAg"
        "ICAgbnAuYXJyYXkoW3JbMV0gZm9yIHIgaW4gcmVzdWx0c10pLA0KICAgICAgICAgICAgbnAuYXJy"
        "YXkoW3JbMl0gZm9yIHIgaW4gcmVzdWx0c10pLA0KICAgICAgICAgICAgbnAuYXJyYXkoW3JbM10g"
        "Zm9yIHIgaW4gcmVzdWx0c10pLA0KICAgICAgICAgICAgW3JbNF0gZm9yIHIgaW4gcmVzdWx0c10s"
        "DQogICAgICAgICkNCg0KICAgIGRlZiBjbG9zZShzZWxmKToNCiAgICAgICAgZm9yIHBpcGUgaW4g"
        "c2VsZi5waXBlczoNCiAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICBwaXBlLnNlbmQo"
        "KCJjbG9zZSIsIE5vbmUpKQ0KICAgICAgICAgICAgZXhjZXB0IE9TRXJyb3I6DQogICAgICAgICAg"
        "ICAgICAgcGFzcw0KICAgICAgICBmb3IgcHJvYyBpbiBzZWxmLnByb2NzOg0KICAgICAgICAgICAg"
        "cHJvYy5qb2luKHRpbWVvdXQ9NSkNCg0KDQpjbGFzcyBTZXF1ZW50aWFsVmVjRW52Og0KICAgICIi"
        "IlNpbmdsZS1wcm9jZXNzIHZlY3Rvcml6ZWQgZW52IGZvciBzaW11bGF0b3JzIHRoYXQgcmVxdWly"
        "ZSBNUEkgKGUuZy4gRmFzdGZhcm0pLg0KDQogICAgUnVucyBhbGwgTiBlbnZzIGluIHRoZSBjYWxs"
        "aW5nIHByb2Nlc3Mgc2VxdWVudGlhbGx5IC0gbm8gbXVsdGlwcm9jZXNzaW5nLA0KICAgIHNvIE1Q"
        "SS5DT01NX1NFTEYuU3Bhd24oKSB3b3JrcyBjb3JyZWN0bHkuDQogICAgIiIiDQoNCiAgICBkZWYg"
        "X19pbml0X18oc2VsZiwgc2VlZHMsIGVudl9pZCk6DQogICAgICAgIGZyb20gd2ZjcmwgaW1wb3J0"
        "IGVudmlyb25tZW50cyBhcyBlbnZzDQogICAgICAgIHNlbGYubiA9IGxlbihzZWVkcykNCiAgICAg"
        "ICAgc2VsZi5lbnZzID0gWw0KICAgICAgICAgICAgZW52cy5tYWtlKA0KICAgICAgICAgICAgICAg"
        "IGVudl9pZCwNCiAgICAgICAgICAgICAgICBtYXhfbnVtX3N0ZXBzPTE1MCwNCiAgICAgICAgICAg"
        "ICAgICBjb250cm9scz17InlhdyI6ICgtNDUsIDQ1LCA1KX0sDQogICAgICAgICAgICAgICAgY29u"
        "dGludW91c19jb250cm9sPVRydWUsDQogICAgICAgICAgICAgICAgbG9nPUZhbHNlLA0KICAgICAg"
        "ICAgICAgKQ0KICAgICAgICAgICAgZm9yIF8gaW4gc2VlZHMNCiAgICAgICAgXQ0KDQogICAgZGVm"
        "IHJlc2V0KHNlbGYsIHNlZWQ9Tm9uZSwgb3B0aW9ucz1Ob25lKToNCiAgICAgICAgc2VlZHMgPSBz"
        "ZWVkIGlmIGlzaW5zdGFuY2Uoc2VlZCwgbGlzdCkgZWxzZSBbc2VlZF0gKiBzZWxmLm4NCiAgICAg"
        "ICAgb2JzX2xpc3QgPSBbDQogICAgICAgICAgICBzZWxmLmVudnNbaV0ucmVzZXQoc2VlZD1zZWVk"
        "c1tpXSwgb3B0aW9ucz1vcHRpb25zKVswXQ0KICAgICAgICAgICAgZm9yIGkgaW4gcmFuZ2Uoc2Vs"
        "Zi5uKQ0KICAgICAgICBdDQogICAgICAgIHJldHVybiB7azogbnAuc3RhY2soW29ba10gZm9yIG8g"
        "aW4gb2JzX2xpc3RdKSBmb3IgayBpbiBvYnNfbGlzdFswXX0sIHt9DQoNCiAgICBkZWYgc3RlcChz"
        "ZWxmLCBhY3Rpb25fZGljdCk6DQogICAgICAgIHJlc3VsdHMgPSBbXQ0KICAgICAgICBmb3IgaSwg"
        "ZW52IGluIGVudW1lcmF0ZShzZWxmLmVudnMpOg0KICAgICAgICAgICAgb2JzLCByZXdhcmQsIHRl"
        "cm1pbmF0ZWQsIHRydW5jYXRlZCwgaW5mbyA9IGVudi5zdGVwKA0KICAgICAgICAgICAgICAgIHtr"
        "OiB2W2ldIGZvciBrLCB2IGluIGFjdGlvbl9kaWN0Lml0ZW1zKCl9DQogICAgICAgICAgICApDQog"
        "ICAgICAgICAgICByZXN1bHRzLmFwcGVuZCgob2JzLCBmbG9hdChucC5zcXVlZXplKHJld2FyZCkp"
        "LCBib29sKHRlcm1pbmF0ZWQpLCBib29sKHRydW5jYXRlZCksIGluZm8pKQ0KICAgICAgICBvYnNf"
        "bGlzdCA9IFtyWzBdIGZvciByIGluIHJlc3VsdHNdDQogICAgICAgIHJldHVybiAoDQogICAgICAg"
        "ICAgICB7azogbnAuc3RhY2soW29ba10gZm9yIG8gaW4gb2JzX2xpc3RdKSBmb3IgayBpbiBvYnNf"
        "bGlzdFswXX0sDQogICAgICAgICAgICBucC5hcnJheShbclsxXSBmb3IgciBpbiByZXN1bHRzXSks"
        "DQogICAgICAgICAgICBucC5hcnJheShbclsyXSBmb3IgciBpbiByZXN1bHRzXSksDQogICAgICAg"
        "ICAgICBucC5hcnJheShbclszXSBmb3IgciBpbiByZXN1bHRzXSksDQogICAgICAgICAgICBbcls0"
        "XSBmb3IgciBpbiByZXN1bHRzXSwNCiAgICAgICAgKQ0KDQogICAgZGVmIGNsb3NlKHNlbGYpOg0K"
        "ICAgICAgICBmb3IgZW52IGluIHNlbGYuZW52czoNCiAgICAgICAgICAgIHRyeToNCiAgICAgICAg"
        "ICAgICAgICBlbnYuY2xvc2UoKQ0KICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAg"
        "ICAgICAgICAgICBwYXNzDQoNCg0KY2xhc3MgTXBpU2VxdWVudGlhbFZlY0VudjoNCiAgICAiIiJT"
        "ZXF1ZW50aWFsIFZlY0VudiBmb3IgTVBJLWJhc2VkIHNpbXVsYXRvcnMgKGUuZy4gRmFzdGZhcm0p"
        "Lg0KDQogICAgV29ya2VyIHJ1bnMgdW5kZXIgYG1waWV4ZWMgLW4gMWAgc28gTVBJLkNPTU1fU0VM"
        "Ri5TcGF3biB3b3JrcyB3aXRob3V0DQogICAgbGF1bmNoaW5nIHRoZSBlbnRpcmUgSnVweXRlciBr"
        "ZXJuZWwgdW5kZXIgbXBpZXhlYy4NCiAgICAiIiINCg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBz"
        "ZWVkcywgZW52X2lkKToNCiAgICAgICAgbXBpZXhlYyA9IHNodXRpbC53aGljaCgnbXBpZXhlYycp"
        "IG9yIHNodXRpbC53aGljaCgnbXNtcGlleGVjJykgb3IgJ21waWV4ZWMnDQogICAgICAgIHdvcmtl"
        "ciA9IG9zLnBhdGguam9pbihvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9f"
        "KSksICdfbXBpX3ZlY193b3JrZXIucHknKQ0KDQogICAgICAgIHNlbGYuX2xpc3RlbmVyID0gTGlz"
        "dGVuZXIoKCdsb2NhbGhvc3QnLCAwKSwgYXV0aGtleT1iJ21waS13b3JrZXInKQ0KICAgICAgICBw"
        "b3J0ID0gc2VsZi5fbGlzdGVuZXIuYWRkcmVzc1sxXQ0KICAgICAgICBzZWxmLl9wcm9jID0gc3Vi"
        "cHJvY2Vzcy5Qb3BlbihbbXBpZXhlYywgJy1uJywgJzEnLCBzeXMuZXhlY3V0YWJsZSwgd29ya2Vy"
        "LCBzdHIocG9ydCldKQ0KICAgICAgICBzZWxmLl9jb25uID0gc2VsZi5fbGlzdGVuZXIuYWNjZXB0"
        "KCkNCiAgICAgICAgc2VsZi5fY29ubi5zZW5kX2J5dGVzKHBpY2tsZS5kdW1wcygoc2VlZHMsIGVu"
        "dl9pZCkpKQ0KICAgICAgICBhc3NlcnQgc2VsZi5fY29ubi5yZWN2X2J5dGVzKCkgPT0gYidyZWFk"
        "eScsICdNUEkgd29ya2VyIGZhaWxlZCB0byBpbml0aWFsaXplJw0KICAgICAgICBzZWxmLm4gPSBs"
        "ZW4oc2VlZHMpDQoNCiAgICBkZWYgcmVzZXQoc2VsZiwgc2VlZD1Ob25lLCBvcHRpb25zPU5vbmUp"
        "Og0KICAgICAgICBzZWVkcyA9IHNlZWQgaWYgaXNpbnN0YW5jZShzZWVkLCBsaXN0KSBlbHNlIFtz"
        "ZWVkXSAqIHNlbGYubg0KICAgICAgICBzZWxmLl9jb25uLnNlbmRfYnl0ZXMocGlja2xlLmR1bXBz"
        "KHsnY21kJzogJ3Jlc2V0JywgJ3NlZWRzJzogc2VlZHMsICdvcHRpb25zJzogb3B0aW9uc30pKQ0K"
        "ICAgICAgICByZXN1bHRzID0gcGlja2xlLmxvYWRzKHNlbGYuX2Nvbm4ucmVjdl9ieXRlcygpKQ0K"
        "ICAgICAgICBvYnNfbGlzdCA9IFtyWzBdIGZvciByIGluIHJlc3VsdHNdDQogICAgICAgIHJldHVy"
        "biB7azogbnAuc3RhY2soW29ba10gZm9yIG8gaW4gb2JzX2xpc3RdKSBmb3IgayBpbiBvYnNfbGlz"
        "dFswXX0sIHt9DQoNCiAgICBkZWYgc3RlcChzZWxmLCBhY3Rpb25fZGljdCk6DQogICAgICAgIHNl"
        "bGYuX2Nvbm4uc2VuZF9ieXRlcyhwaWNrbGUuZHVtcHMoeydjbWQnOiAnc3RlcCcsICdhY3Rpb24n"
        "OiBhY3Rpb25fZGljdH0pKQ0KICAgICAgICByZXN1bHRzID0gcGlja2xlLmxvYWRzKHNlbGYuX2Nv"
        "bm4ucmVjdl9ieXRlcygpKQ0KICAgICAgICBvYnNfbGlzdCA9IFtyWzBdIGZvciByIGluIHJlc3Vs"
        "dHNdDQogICAgICAgIHJldHVybiAoDQogICAgICAgICAgICB7azogbnAuc3RhY2soW29ba10gZm9y"
        "IG8gaW4gb2JzX2xpc3RdKSBmb3IgayBpbiBvYnNfbGlzdFswXX0sDQogICAgICAgICAgICBucC5h"
        "cnJheShbclsxXSBmb3IgciBpbiByZXN1bHRzXSksDQogICAgICAgICAgICBucC5hcnJheShbclsy"
        "XSBmb3IgciBpbiByZXN1bHRzXSksDQogICAgICAgICAgICBucC5hcnJheShbclszXSBmb3IgciBp"
        "biByZXN1bHRzXSksDQogICAgICAgICAgICBbcls0XSBmb3IgciBpbiByZXN1bHRzXSwNCiAgICAg"
        "ICAgKQ0KDQogICAgZGVmIGNsb3NlKHNlbGYpOg0KICAgICAgICB0cnk6DQogICAgICAgICAgICBz"
        "ZWxmLl9jb25uLnNlbmRfYnl0ZXMocGlja2xlLmR1bXBzKHsnY21kJzogJ2Nsb3NlJ30pKQ0KICAg"
        "ICAgICAgICAgc2VsZi5fY29ubi5yZWN2X2J5dGVzKCkNCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlv"
        "bjoNCiAgICAgICAgICAgIHBhc3MNCiAgICAgICAgc2VsZi5fY29ubi5jbG9zZSgpDQogICAgICAg"
        "IHNlbGYuX2xpc3RlbmVyLmNsb3NlKCkNCiAgICAgICAgc2VsZi5fcHJvYy53YWl0KHRpbWVvdXQ9"
        "MTApDQo="
    )
    _dst = '/tmp/wfcrl_vec_utils.py'
    with open(_dst, 'wb') as _f:
        _f.write(base64.b64decode(_MODULE_B64))
    _module_dir = '/tmp'
    print(f"Wrote embedded wfcrl_vec_utils.py to {_dst}")

if _module_dir not in sys.path:
    sys.path.insert(0, _module_dir)

from wfcrl_vec_utils import ParallelVecEnv, SequentialVecEnv, MpiSequentialVecEnv

N_ENVS = ROLLOUTS_PER_UPDATE
print(f"VecEnv import OK; will spawn {N_ENVS} workers per env_id during sweep")


Wrote embedded wfcrl_vec_utils.py to /tmp/wfcrl_vec_utils.py
VecEnv import OK; will spawn 16 workers per env_id during sweep


## Save/Load Checkpoints

In [42]:
import json
from datetime import datetime


def make_run_dir(env_id):
    """Per-env_id checkpoint directory: checkpoints/{env_id}/run_{timestamp}/"""
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = os.path.join("checkpoints", env_id, f"run_{ts}")
    os.makedirs(path, exist_ok=True)
    return path


def save_checkpoint(path, policy, optimizer, update, episode_rewards, loss_log, hparams_dict):
    os.makedirs(path, exist_ok=True)
    torch.save({
        "policy_state":    policy.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "update":          update,
        "episode_rewards": episode_rewards,
        "loss_log":        loss_log,
    }, os.path.join(path, "checkpoint.pt"))
    with open(os.path.join(path, "hparams.json"), "w") as f:
        json.dump(hparams_dict, f, indent=2)


def load_checkpoint(path, policy, optimizer):
    ckpt = torch.load(os.path.join(path, "checkpoint.pt"), map_location=DEVICE)
    policy.load_state_dict(ckpt["policy_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    print(f"Loaded checkpoint: update {ckpt['update']}, "
          f"{len(ckpt['episode_rewards'])} episodes logged")
    return ckpt["update"], ckpt["episode_rewards"], ckpt["loss_log"]


HPARAMS_DICT = {
    "SEED": SEED, "N_EPISODES": N_EPISODES, "N_ENVS": N_ENVS,
    "LR": LR, "MIN_LR": MIN_LR, "N_EPOCHS": N_EPOCHS, "BATCH_SIZE": BATCH_SIZE,
    "GAMMA": GAMMA, "GAE_LAMBDA": GAE_LAMBDA, "CLIP_EPS": CLIP_EPS,
    "VF_COEF": VF_COEF, "ENT_COEF_START": ENT_COEF_START, "ENT_COEF_END": ENT_COEF_END,
    "MAX_GRAD": MAX_GRAD, "REWARD_SCALE": REWARD_SCALE,
    "TERMINAL_SCALE": TERMINAL_SCALE, "CONVERGENCE_COEF": CONVERGENCE_COEF,
    "MAX_STEPS": MAX_STEPS,
}


## Compute_gae

In [43]:
def compute_gae(rewards, values, dones, gamma=GAMMA, lam=GAE_LAMBDA):
    """
    Generalized Advantage Estimation (GAE-lambda).
    Episode always terminates, so bootstrap value = 0.
    """
    T          = len(rewards)
    advantages = np.zeros(T, dtype=np.float32)
    gae        = 0.0
    next_value = 0.0
    for t in reversed(range(T)):
        mask           = 1.0 - dones[t]
        delta          = rewards[t] + gamma * next_value * mask - values[t]
        gae            = delta + gamma * lam * mask * gae
        advantages[t]  = gae
        next_value     = values[t]
    returns = advantages + np.array(values, dtype=np.float32)
    return advantages, returns

## PPO_update

In [44]:
def ppo_update(obs_buf, act_buf, logp_buf, adv_buf, ret_buf, ent_coef=ENT_COEF):
    """Mini-batch PPO update over N_EPOCHS passes."""
    obs_t  = torch.tensor(obs_buf,  dtype=torch.float32, device=DEVICE)
    act_t  = torch.tensor(act_buf,  dtype=torch.float32, device=DEVICE)
    logp_t = torch.tensor(logp_buf, dtype=torch.float32, device=DEVICE)
    adv_t  = torch.tensor(adv_buf,  dtype=torch.float32, device=DEVICE)
    ret_t  = torch.tensor(ret_buf,  dtype=torch.float32, device=DEVICE)

    adv_t = (adv_t - adv_t.mean()) / (adv_t.std() + 1e-8)  # normalize advantages

    N = len(obs_t)
    total_pg = total_vf = total_ent = 0.0
    n_batches = 0

    for _ in range(N_EPOCHS):
        for b in torch.randperm(N, device=DEVICE).split(BATCH_SIZE):
            new_logp, entropy, new_val = policy.evaluate(obs_t[b], act_t[b])

            ratio    = (new_logp - logp_t[b]).exp()
            pg_loss  = -torch.min(
                ratio * adv_t[b],
                torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv_t[b]
            ).mean()
            vf_loss  = ((new_val - ret_t[b]) ** 2).mean()
            ent_loss = -entropy.mean()

            loss = pg_loss + VF_COEF * vf_loss + ent_coef * ent_loss
            if not torch.isfinite(loss):
                optimizer.zero_grad()
                continue

            loss.backward()
            grad_norm = nn.utils.clip_grad_norm_(policy.parameters(), MAX_GRAD)
            if not torch.isfinite(grad_norm):
                optimizer.zero_grad()
                continue
            optimizer.step()
            optimizer.zero_grad()

            total_pg  += pg_loss.item()
            total_vf  += vf_loss.item()
            total_ent += ent_loss.item()
            n_batches += 1

    if n_batches == 0:
        return {"pg": 0.0, "vf": 0.0, "ent": 0.0}
    return {"pg": total_pg / n_batches,
            "vf": total_vf / n_batches,
            "ent": total_ent / n_batches}

## Evaluation: Greedy Policy Rollout

Run the trained policy deterministically (actor mean only, no sampling noise).

In [45]:
def evaluate_policy(n_episodes: int = 5, greedy: bool = True, seed_offset: int = 9001):
    """Return episode reward list and a DataFrame of the first episode's trajectory.
    Reads module globals: policy, env, env_reset, flatten_obs, WIND_OPTS, N_TURBINES, DEVICE."""
    policy.eval()
    rewards   = []
    eval_rows = []

    with torch.no_grad():
        for ep in range(n_episodes):
            obs_dict = env_reset(seed_offset + ep, WIND_OPTS)
            done, total_r, step = False, 0.0, 0

            while not done:
                obs_np   = flatten_obs(obs_dict, step=step)
                obs_t    = torch.tensor(obs_np, dtype=torch.float32,
                                        device=DEVICE).unsqueeze(0)
                if greedy:
                    features = policy.actor_backbone(obs_t)
                    action_t = policy.actor_mean(features)
                else:
                    action_t = policy.act(obs_t)[0]
                action_np    = np.clip(
                    action_t.squeeze(0).cpu().numpy(),
                    env.action_space["yaw"].low,
                    env.action_space["yaw"].high,
                )
                joint_action = {"yaw": action_np}

                obs_dict, reward, termination, truncation, info = env.step(joint_action)
                r        = float(reward[0]) if hasattr(reward, "__len__") else float(reward)
                total_r += r

                if ep == 0:
                    eval_rows.append({
                        "step":   step,
                        "reward": r,
                        **{f"yaw_{i}":   obs_dict["yaw"][i]   for i in range(N_TURBINES)},
                        **{f"power_{i}": info["power"][i]     for i in range(N_TURBINES)},
                    })
                step += 1
                done = termination or truncation

            rewards.append(total_r)

    policy.train()
    return rewards, pd.DataFrame(eval_rows)


In [46]:
def evaluate_no_control(n_episodes: int = 180, seed_offset: int = 9001):
    """Baseline: always send zero yaw deltas (no steering applied).
    Reads module globals: env, env_reset, WIND_OPTS, N_TURBINES."""
    zero_action = {"yaw": np.zeros(N_TURBINES, dtype=np.float32)}
    rewards = []
    baseline_rows = []

    for ep in range(n_episodes):
        obs_dict = env_reset(seed_offset + ep, WIND_OPTS)
        done, total_r, step = False, 0.0, 0
        while not done:
            obs_dict, reward, termination, truncation, info = env.step(zero_action)
            r = float(reward[0]) if hasattr(reward, "__len__") else float(reward)
            total_r += r
            if ep == 0:
                baseline_rows.append({
                    "step": step,
                    "reward": r,
                    **{f"power_{i}": info["power"][i] for i in range(N_TURBINES)},
                })
            step += 1
            done = termination or truncation
        rewards.append(total_r)

    return rewards, pd.DataFrame(baseline_rows)


## PPO Training Sweep

In [ ]:
# ============================================================
# Training sweep across all env_ids
# ============================================================
# Per-env state is snapshotted into these dicts after each successful run so
# every trained policy stays in scope - the SELECTED_ENV cell below picks one
# to visualize, and the cross-env summary cells at the bottom plot all of them.

policies        = {}   # env_id -> trained ActorCritic
envs_dict       = {}   # env_id -> single-process eval env (kept open)
training_curves = {}   # env_id -> dict(episode_rewards, loss_log)
results         = {}   # env_id -> dict(eval_mean, eval_std, baseline_mean, gain_pct, dfs, ...)

for sweep_env_id in ENV_IDS:
    print(f"\n{'='*70}\nTraining: {sweep_env_id}\n{'='*70}")
    vec_env = None
    try:
        # ---- Single-process eval env -----------------------------------------
        env = envs.make(
            env_id=sweep_env_id,
            max_num_steps=150,
            controls={"yaw": (-45, 45, 5)},
            continuous_control=True,
            log=True,
        )
        N_TURBINES = env.num_turbines
        _OBS_LOW   = np.concatenate([env.observation_space[k].low  for k in _OBS_KEYS]).astype(np.float32)
        _OBS_HIGH  = np.concatenate([env.observation_space[k].high for k in _OBS_KEYS]).astype(np.float32)
        OBS_DIM    = _OBS_LOW.shape[0] + 1   # +1 for the appended timestep feature
        ACT_DIM    = N_TURBINES
        print(f"  N_TURBINES={N_TURBINES}, OBS_DIM={OBS_DIM}, ACT_DIM={ACT_DIM}")

        # ---- Fresh policy / optimizer ----------------------------------------
        policy    = ActorCritic(OBS_DIM, ACT_DIM, hidden_dim=256).to(DEVICE)
        optimizer = optim.Adam(policy.parameters(), lr=LR, eps=1e-5)
        episode_rewards = []
        loss_log = {"pg": [], "vf": [], "ent": []}
        start_update = 0
        SAVE_DIR = make_run_dir(sweep_env_id)
        print(f"  Saving to {SAVE_DIR}")

        # ---- Rollout env (sequential for Fastfarm to avoid MPI/spawn conflict)
        VecEnvCls = MpiSequentialVecEnv if "Fastfarm" in sweep_env_id else ParallelVecEnv
        vec_env = VecEnvCls([SEED + i for i in range(N_ENVS)], env_id=sweep_env_id)

        # ---- Per-env baseline cache (zero-yaw rewards per training seed) -----
        BASELINE_CACHE = f"baseline_cache_{sweep_env_id}_S{SEED}_E{N_EPISODES}_N{N_ENVS}.npz"
        if os.path.exists(BASELINE_CACHE):
            _data = np.load(BASELINE_CACHE)
            baseline_step_rewards = {int(k): list(_data[k]) for k in _data.files}
            print(f"  Loaded baseline cache ({len(baseline_step_rewards)} seeds)")
        else:
            baseline_step_rewards = {}
            _zero_action = {"yaw": np.zeros((N_ENVS, ACT_DIM), dtype=np.float32)}
            print(f"  Pre-computing baseline for {N_EPISODES * N_ENVS} seeds...")
            t0 = time.time()
            for _update in range(N_EPISODES):
                seeds = [SEED + _update * N_ENVS + _i for _i in range(N_ENVS)]
                obs_batch, _ = vec_env.reset(seed=seeds, options=WIND_OPTS)
                step_rewards = [[] for _ in range(N_ENVS)]
                dones = np.zeros(N_ENVS, dtype=bool)
                baseline_step = 0
                while not dones.all():
                    obs_batch, rewards, terminations, truncations, _ = vec_env.step(_zero_action)
                    for i in range(N_ENVS):
                        if not dones[i]:
                            step_rewards[i].append(float(np.squeeze(rewards[i])))
                            dones[i] = bool(terminations[i]) or bool(truncations[i])
                    baseline_step += 1
                    print_status(
                        f"    baseline batch {_update + 1}/{N_EPISODES} "
                        f"| step {baseline_step}/{MAX_STEPS} "
                        f"| done {dones.sum()}/{N_ENVS}"
                    )
                for i, seed in enumerate(seeds):
                    baseline_step_rewards[seed] = step_rewards[i]
                print_status(
                    f"    baseline {_update + 1}/{N_EPISODES} batches done "
                    f"({time.time()-t0:.1f}s)"
                )
            print()
            np.savez(BASELINE_CACHE, **{str(k): np.array(v) for k, v in baseline_step_rewards.items()})
            print(f"  Saved baseline cache ({time.time()-t0:.1f}s)")

        # ---- Training loop ---------------------------------------------------
        ACT_LOW  = env.action_space["yaw"].low
        ACT_HIGH = env.action_space["yaw"].high
        remaining = N_EPISODES - start_update
        print(f"  Training: {remaining} updates x {N_ENVS} envs x {MAX_STEPS} steps "
              f"~= {remaining * N_ENVS * MAX_STEPS:,} steps")

        for update in range(start_update, N_EPISODES):
            frac = 1.0 - update / N_EPISODES
            for pg in optimizer.param_groups:
                pg["lr"] = max(LR * frac, MIN_LR)
            frac_ent = update / max(N_EPISODES - 1, 1)
            ent_coef = ENT_COEF_START + frac_ent * (ENT_COEF_END - ENT_COEF_START)

            obs_bufs  = [[] for _ in range(N_ENVS)]
            act_bufs  = [[] for _ in range(N_ENVS)]
            logp_bufs = [[] for _ in range(N_ENVS)]
            rew_bufs  = [[] for _ in range(N_ENVS)]
            val_bufs  = [[] for _ in range(N_ENVS)]
            done_bufs = [[] for _ in range(N_ENVS)]
            ep_rewards = np.zeros(N_ENVS)

            seeds = [SEED + update * N_ENVS + i for i in range(N_ENVS)]
            obs_batch, _ = vec_env.reset(seed=seeds, options=WIND_OPTS)
            done_mask     = np.zeros(N_ENVS, dtype=bool)
            step_counters = np.zeros(N_ENVS, dtype=np.int32)

            while not done_mask.all():
                obs_np = flatten_obs_batch(obs_batch, steps=step_counters)
                obs_t  = torch.tensor(obs_np, dtype=torch.float32, device=DEVICE)
                with torch.no_grad():
                    action_t, logp_t, val_t = policy.act(obs_t)
                action_np = np.clip(action_t.cpu().numpy(), ACT_LOW, ACT_HIGH)
                obs_batch, rewards, terminations, truncations, _ = vec_env.step({"yaw": action_np})

                for i in range(N_ENVS):
                    if not done_mask[i]:
                        obs_bufs[i].append(obs_np[i])
                        act_bufs[i].append(action_np[i])
                        logp_bufs[i].append(logp_t[i].item())
                        r = float(np.squeeze(rewards[i]))
                        ep_rewards[i] += r

                        step_idx = len(rew_bufs[i])
                        bl = baseline_step_rewards[seeds[i]][step_idx]
                        diff_reward = (r - bl) / REWARD_SCALE

                        t_frac = step_idx / MAX_STEPS
                        convergence_penalty = CONVERGENCE_COEF * (t_frac ** 2) * float(np.mean(np.abs(action_np[i])))
                        diff_reward -= convergence_penalty

                        is_done = bool(terminations[i]) or bool(truncations[i])
                        if is_done:
                            diff_reward += TERMINAL_SCALE * (r - bl) / REWARD_SCALE

                        rew_bufs[i].append(diff_reward)
                        val_bufs[i].append(val_t[i].item())
                        done_bufs[i].append(float(is_done))
                        step_counters[i] += 1
                        if is_done:
                            done_mask[i] = True
                print_status(
                    f"  Training {sweep_env_id} | update {update+1}/{N_EPISODES} "
                    f"| rollout step {int(step_counters.max())}/{MAX_STEPS} "
                    f"| done {done_mask.sum()}/{N_ENVS} "
                    f"| reward mean={ep_rewards.mean():.2f}"
                )

            obs_all, act_all, logp_all, adv_all, ret_all = [], [], [], [], []
            for i in range(N_ENVS):
                adv, ret = compute_gae(rew_bufs[i], val_bufs[i], done_bufs[i])
                obs_all.append(np.array(obs_bufs[i]))
                act_all.append(np.array(act_bufs[i]))
                logp_all.append(np.array(logp_bufs[i]))
                adv_all.append(adv)
                ret_all.append(ret)
                episode_rewards.append(ep_rewards[i])

            print_status(
                f"  Training {sweep_env_id} | update {update+1}/{N_EPISODES} "
                f"| rollout done, running PPO update"
            )
            losses = ppo_update(
                np.concatenate(obs_all), np.concatenate(act_all),
                np.concatenate(logp_all), np.concatenate(adv_all),
                np.concatenate(ret_all), ent_coef=ent_coef,
            )
            for k, v in losses.items():
                loss_log[k].append(v)
            
            print_status(
                f"  Training {sweep_env_id} | update {update+1}/{N_EPISODES} "
                f"| mean reward={ep_rewards.mean():.2f} "
                f"| pg={losses['pg']:.4f} vf={losses['vf']:.6f}"
            )

            print()
            if (update + 1) % EVAL_EVERY == 0:
                cur_lr = optimizer.param_groups[0]["lr"]
                print(f"  Update {update+1:3d}/{N_EPISODES} | lr={cur_lr:.2e} ent={ent_coef:.4f} "
                      f"| mean reward: {ep_rewards.mean():.2f} "
                      f"| pg={losses['pg']:.4f}  vf={losses['vf']:.6f}")

            if (update + 1) % 10 == 0:
                print_status(
                    f"  Training {sweep_env_id} | update {update+1}/{N_EPISODES} "
                    f"| saving checkpoint"
                )
                save_checkpoint(SAVE_DIR, policy, optimizer, update + 1,
                                episode_rewards, loss_log, HPARAMS_DICT)

        save_checkpoint(SAVE_DIR, policy, optimizer, N_EPISODES,
                        episode_rewards, loss_log, HPARAMS_DICT)

        vec_env.close()
        vec_env = None

        # ---- Evaluation ------------------------------------------------------
        n_eval = N_EVAL
        print_status(
            f"  Evaluating {sweep_env_id} | PPO policy | episodes={n_eval}",
            force=True,
        )
        print()
        eval_rewards, eval_df = evaluate_policy(n_episodes=n_eval)
        print_status(
            f"  Evaluating {sweep_env_id} | no-control baseline "
            f"| episodes={max(N_EVAL, n_eval // 3)}",
            force=True,
        )
        print()
        baseline_rewards, baseline_df = evaluate_no_control(n_episodes=max(N_EVAL, n_eval // 3))
        eval_df["total_power"]     = sum(eval_df[f"power_{i}"]     for i in range(N_TURBINES))
        baseline_df["total_power"] = sum(baseline_df[f"power_{i}"] for i in range(N_TURBINES))

        # ---- Snapshot --------------------------------------------------------
        policies[sweep_env_id]        = policy
        envs_dict[sweep_env_id]       = env
        training_curves[sweep_env_id] = {"episode_rewards": list(episode_rewards),
                                         "loss_log": {k: list(v) for k, v in loss_log.items()}}
        results[sweep_env_id] = {
            "eval_rewards":    eval_rewards,
            "eval_df":         eval_df,
            "baseline_rewards": baseline_rewards,
            "baseline_df":     baseline_df,
            "eval_mean":       float(np.mean(eval_rewards)),
            "eval_std":        float(np.std(eval_rewards)),
            "baseline_mean":   float(np.mean(baseline_rewards)),
            "baseline_std":    float(np.std(baseline_rewards)),
            "gain_pct":        100.0 * (np.mean(eval_rewards) - np.mean(baseline_rewards))
                               / abs(np.mean(baseline_rewards) + 1e-9),
            "n_turbines":      N_TURBINES,
            "obs_low":         _OBS_LOW.copy(),
            "obs_high":        _OBS_HIGH.copy(),
            "save_dir":        SAVE_DIR,
        }
        print(f"  DONE: PPO={results[sweep_env_id]['eval_mean']:.2f} +- "
              f"{results[sweep_env_id]['eval_std']:.2f}, "
              f"baseline={results[sweep_env_id]['baseline_mean']:.2f}, "
              f"gain={results[sweep_env_id]['gain_pct']:+.1f}%")

    except Exception as exc:
        print(f"  FAILED ({type(exc).__name__}): {exc}")
        results[sweep_env_id] = {"error": f"{type(exc).__name__}: {exc}"}
        if vec_env is not None:
            try:
                vec_env.close()
            except Exception:
                pass

print(f"\n{'='*70}\nSweep complete\n{'='*70}")
n_ok = sum(1 for r in results.values() if "error" not in r)
print(f"  {n_ok}/{len(ENV_IDS)} envs trained successfully")
for env_id_, r in results.items():
    if "error" in r:
        print(f"  {env_id_}: FAILED - {r['error']}")
    else:
        print(f"  {env_id_}: PPO={r['eval_mean']:.2f}+-{r['eval_std']:.2f}, "
              f"baseline={r['baseline_mean']:.2f}, gain={r['gain_pct']:+.1f}%")


Training: Turb_TCRWP_Floris
  N_TURBINES=32, OBS_DIM=99, ACT_DIM=32
  Saving to checkpoints/Turb_TCRWP_Floris/run_20260501_232421


## Per-Environment Replay & Visualization

Pick any env from the sweep to inspect. The downstream visualization cells operate on whichever `policy`/`env`/`eval_df`/etc. the selector cell most recently set, so just change `SELECTED_ENV` and re-run the cells below.

In [ ]:
# Pick which trained env to visualize. Must be a key of `policies`.
SELECTED_ENV = "Turb3_Row1_Floris"   # change me

if SELECTED_ENV not in policies:
    raise KeyError(f"{SELECTED_ENV} not in policies; available: {list(policies.keys())}")

# Re-bind module-level names so the existing visualization / evaluation cells
# (which read `policy`, `env`, `eval_df`, etc. as globals) work unchanged.
policy          = policies[SELECTED_ENV]
env             = envs_dict[SELECTED_ENV]
N_TURBINES      = results[SELECTED_ENV]["n_turbines"]
_OBS_LOW        = results[SELECTED_ENV]["obs_low"]
_OBS_HIGH       = results[SELECTED_ENV]["obs_high"]
OBS_DIM         = _OBS_LOW.shape[0] + 1
ACT_DIM         = N_TURBINES

eval_rewards    = results[SELECTED_ENV]["eval_rewards"]
eval_df         = results[SELECTED_ENV]["eval_df"]
baseline_rewards = results[SELECTED_ENV]["baseline_rewards"]
baseline_df     = results[SELECTED_ENV]["baseline_df"]
episode_rewards = training_curves[SELECTED_ENV]["episode_rewards"]
loss_log        = training_curves[SELECTED_ENV]["loss_log"]

print(f"Now viewing: {SELECTED_ENV}")
print(f"  N_TURBINES={N_TURBINES}, eval mean={np.mean(eval_rewards):.2f} +- {np.std(eval_rewards):.2f}")
print(f"  baseline mean={np.mean(baseline_rewards):.2f}, gain={results[SELECTED_ENV]['gain_pct']:+.1f}%")


## Learning Curves

In [ ]:
WINDOW   = 20
smoothed = np.convolve(episode_rewards, np.ones(WINDOW) / WINDOW, mode="valid")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(episode_rewards, alpha=0.25, color="steelblue")
ax.plot(np.arange(WINDOW - 1, len(episode_rewards)), smoothed,
        color="steelblue", linewidth=2, label=f"{WINDOW}-ep avg")
ax.set_xlabel("Episode")
ax.set_ylabel("Total reward")
ax.set_title("PPO Training Reward (Sc.2 - Random Wind)")
ax.legend()

ax = axes[1]
ax.plot(loss_log["pg"],  label="Policy (PG) loss")
ax.plot(loss_log["vf"],  label="Value loss")
ax.set_xlabel("Episode")
ax.set_ylabel("Loss")
ax.set_title("PPO Losses")
ax.legend()

plt.tight_layout()
plt.show()

## Yaw Angles and Power Over a Representative Episode

In [ ]:
colors = sns.color_palette("tab10", N_TURBINES)
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

# -- Yaw angles ----------------------------------------------------------------
ax = axes[0]
for i in range(N_TURBINES):
    ax.plot(eval_df["step"], eval_df[f"yaw_{i}"], color=colors[i], label=f"Turbine {i}")
    final_yaw  = eval_df[f"yaw_{i}"].iloc[-1]
    final_step = eval_df["step"].iloc[-1]
    ax.annotate(f"{final_yaw:.1f}",
                xy=(final_step, final_yaw),
                xytext=(6, 0), textcoords="offset points",
                color=colors[i], fontsize=9, va="center", fontweight="bold")
ax.set_ylabel("Yaw angle (deg)")
ax.set_title("PPO Policy - Yaw Angles")
ax.legend(loc="upper left")

# -- Power output --------------------------------------------------------------
ax = axes[1]
for i in range(N_TURBINES):
    ax.plot(eval_df["step"], eval_df[f"power_{i}"], color=colors[i], label=f"Turbine {i}")
ax.set_ylabel("Power (MW)")
ax.set_title("PPO Policy - Power Output")
ax.legend(loc="upper left")

# -- Step reward ---------------------------------------------------------------
ax = axes[2]
ax.plot(eval_df["step"], eval_df["reward"], color="steelblue", linewidth=1.5)
ax.set_xlabel("Step")
ax.set_ylabel("Step reward")
ax.set_title("PPO Policy - Step Reward")

# -- Convergence marker: first step (after step 20) where rolling max yaw rate < 0.1 deg/step
yaw_rate = (
    eval_df[[f"yaw_{i}" for i in range(N_TURBINES)]]
    .diff().abs().max(axis=1)
    .rolling(10, min_periods=1).mean()
)
conv_candidates = eval_df["step"][(yaw_rate < 0.1) & (eval_df["step"] > 20)]
if not conv_candidates.empty:
    conv_step = int(conv_candidates.iloc[0])
    yaw_max = eval_df[[f"yaw_{i}" for i in range(N_TURBINES)]].values.max()
    for a in axes:
        a.axvline(conv_step, color="gray", linestyle="--", alpha=0.6, linewidth=1)
    axes[0].text(conv_step + 2, yaw_max * 0.85, "yaw stabilized", fontsize=8, color="gray")

plt.tight_layout()
plt.show()


## No-control

## No-Control Baseline & Total Farm Power

Compare the PPO policy against a zero-action baseline (no yaw steering) on the same episodes. Total farm power = sum across all turbines, which is the actual metric the policy optimizes.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(eval_df["step"], eval_df["total_power"],
        color="steelblue", linewidth=2, label="PPO policy")
ax.plot(baseline_df["step"], baseline_df["total_power"],
        color="gray", linestyle="--", linewidth=1.5, label="No control (baseline)")
ax.fill_between(eval_df["step"],
                baseline_df["total_power"], eval_df["total_power"],
                alpha=0.15, color="steelblue", label="PPO gain region")

mean_ppo  = eval_df["total_power"].mean()
mean_base = baseline_df["total_power"].mean()
ax.axhline(mean_ppo,  color="steelblue", linestyle=":", alpha=0.7, linewidth=1)
ax.axhline(mean_base, color="gray",      linestyle=":", alpha=0.7, linewidth=1)
x_label = eval_df["step"].iloc[-1] * 0.02
ax.text(x_label, mean_ppo  + 0.0005, f"PPO mean: {mean_ppo:.4f} MW",      color="steelblue", fontsize=9)
ax.text(x_label, mean_base - 0.0012, f"Baseline mean: {mean_base:.4f} MW", color="gray",      fontsize=9)

ax.set_xlabel("Step")
ax.set_ylabel("Total farm power (MW)")
ax.set_title("Total Farm Power: PPO vs No-Control Baseline")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
eval_df["cumulative_energy"]     = eval_df["total_power"].cumsum()
baseline_df["cumulative_energy"] = baseline_df["total_power"].cumsum()

final_ppo  = eval_df["cumulative_energy"].iloc[-1]
final_base = baseline_df["cumulative_energy"].iloc[-1]
gain_pct   = (final_ppo - final_base) / final_base * 100

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(eval_df["step"], eval_df["cumulative_energy"],
        color="steelblue", linewidth=2,
        label=f"PPO policy  ({final_ppo:.1f} MW*steps)")
ax.plot(baseline_df["step"], baseline_df["cumulative_energy"],
        color="gray", linestyle="--", linewidth=1.5,
        label=f"No control  ({final_base:.1f} MW*steps)")
ax.fill_between(eval_df["step"],
                baseline_df["cumulative_energy"], eval_df["cumulative_energy"],
                alpha=0.15, color="steelblue")
ax.set_xlabel("Step")
ax.set_ylabel("Cumulative energy (MW * steps)")
ax.set_title(f"Cumulative Farm Energy Harvested  (PPO gain: {gain_pct:+.2f}%)")
ax.legend()
plt.tight_layout()
plt.show()


## Yaw Angle vs. Power Scatter

Each point is one step of the evaluated episode, colored by step number (dark = early, bright = late). Shows which yaw offsets the policy explored and the resulting power tradeoff per turbine.

In [ ]:
fig, axes = plt.subplots(1, N_TURBINES, figsize=(14, 4))

for i, ax in enumerate(axes):
    sc = ax.scatter(
        eval_df[f"yaw_{i}"], eval_df[f"power_{i}"],
        c=eval_df["step"], cmap="viridis", s=25, alpha=0.8, edgecolors="none",
    )
    ax.set_xlabel("Yaw angle (deg)")
    ax.set_ylabel("Power (MW)")
    ax.set_title(f"Turbine {i}")
    plt.colorbar(sc, ax=ax, label="Step")

plt.suptitle("Yaw Angle vs. Power Output per Turbine  (color = step)", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:

for i in range(N_TURBINES):
    print(f"{i}: {eval_df['step'].iloc[0]}, {eval_df[f'yaw_{i}'].iloc[0]}")

In [ ]:
# print the last step's yaw angles for each turbine
n_turbines = len([c for c in eval_df.columns if c.startswith("yaw_")])
for i in range(n_turbines):
    print(f"{i}: {eval_df['step'].iloc[-1]}, {eval_df[f'yaw_{i}'].iloc[-1]}")

## Final Comparison

In [ ]:
# WFCRL Table 7 reference (Turb3_Row1_Floris, FLORIS, 200k steps, eval on Sc.2).
# Only meaningful when SELECTED_ENV == "Turb3_Row1_Floris"; for other envs the
# WFCRL bars are kept as a reference scale but are not directly comparable.
WFCRL_BASELINES = {
    "IPPO/Sc.2 (WFCRL)":  (406.0, 10.5),
    "MAPPO/Sc.2 (WFCRL)": (372.4, 15.5),
}

my_results = {
    f"PPO ({SELECTED_ENV})": (np.mean(eval_rewards), np.std(eval_rewards)),
    "No-control baseline":   (np.mean(baseline_rewards), np.std(baseline_rewards)),
}

all_labels = list(WFCRL_BASELINES.keys()) + list(my_results.keys())
all_means  = [v[0] for v in WFCRL_BASELINES.values()] + [v[0] for v in my_results.values()]
all_errs   = [v[1] for v in WFCRL_BASELINES.values()] + [v[1] for v in my_results.values()]
colors     = ["#4a90e2"] * len(WFCRL_BASELINES) + ["#2ca02c", "#7f7f7f"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(all_labels, all_means, yerr=all_errs, color=colors, capsize=6,
              edgecolor="black", linewidth=0.8)
ax.set_ylabel("Mean episode reward (Sc.2)")
ax.set_title(f"Method Comparison - {SELECTED_ENV} (Sc.2: Random Wind)\nvs. WFCRL arXiv:2501.13592 Table 7")
for b, m in zip(bars, all_means):
    ax.text(b.get_x() + b.get_width() / 2, m + 4, f"{m:.1f}", ha="center", fontsize=10)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## Wind Farm Layout with Yaw Angles

In [ ]:
# NOTE: Turbine positions and layout are hardcoded for `Turb3_Row1_Floris`
# (3 turbines at 4D spacing along x). For other env_ids the visualization will
# only render the first 3 turbines and may not reflect the true farm geometry.
if SELECTED_ENV != "Turb3_Row1_Floris":
    print(f"Layout is hardcoded for Turb3_Row1_Floris; SELECTED_ENV={SELECTED_ENV} "
          f"has {N_TURBINES} turbines - skipping farm-layout viz.")
else:
    TURB_X = np.array([0.0, 504.0, 1008.0])
    TURB_Y = np.array([0.0, 0.0, 0.0])
    D = 126.0  # rotor diameter (m)

    def plot_farm_layout_with_yaw(yaw_angles, wind_speed, wind_dir_deg, ax=None, step=None):
        # wind_dir_deg: meteorological FROM-direction (270 = from west, blows east = +x)
        # yaw_angles: offset in degrees per turbine from the wind direction
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 4))

        flow_math_deg = (270 - wind_dir_deg) % 360
        flow_rad = np.deg2rad(flow_math_deg)

        arrow_len = 180.0
        for row_y in np.linspace(-280, 280, 5):
            ax.annotate(
                '', xy=(TURB_X[-1] + 300 + arrow_len * np.cos(flow_rad),
                        row_y + arrow_len * np.sin(flow_rad)),
                xytext=(TURB_X[-1] + 300, row_y),
                arrowprops=dict(arrowstyle='->', color='lightblue', lw=1.5),
            )

        colors = sns.color_palette('tab10', N_TURBINES)
        for i, (tx, ty, yaw) in enumerate(zip(TURB_X, TURB_Y, yaw_angles)):
            circle = plt.Circle((tx, ty), D / 2, color=colors[i], alpha=0.2, zorder=2)
            ax.add_patch(circle)
            ax.plot(tx, ty, 'o', color=colors[i], ms=8, zorder=3)

            rotor_rad = np.deg2rad(flow_math_deg + yaw)
            perp_rad  = rotor_rad + np.pi / 2
            blade_len = D / 2 * 0.88
            ax.plot(
                [tx - blade_len * np.cos(perp_rad), tx + blade_len * np.cos(perp_rad)],
                [ty - blade_len * np.sin(perp_rad), ty + blade_len * np.sin(perp_rad)],
                color=colors[i], lw=3, zorder=3,
            )
            ax.annotate(
                '', xy=(tx + 75 * np.cos(rotor_rad), ty + 75 * np.sin(rotor_rad)),
                xytext=(tx, ty),
                arrowprops=dict(arrowstyle='->', color=colors[i], lw=2),
            )
            ax.text(tx, ty - D * 0.78, f'T{i}\nyaw={yaw:.1f}deg',
                    ha='center', fontsize=9, color=colors[i], fontweight='bold')

        ax.set_xlim(-300, TURB_X[-1] + 520)
        ax.set_ylim(-400, 400)
        ax.set_aspect('equal')
        ax.set_xlabel('x (m)')
        ax.set_ylabel('y (m)')
        title = f'Wind farm  |  Wind: {wind_speed:.1f} m/s from {wind_dir_deg:.0f} deg'
        if step is not None:
            title += f'  |  Step {step}'
        ax.set_title(f'{title}  ({SELECTED_ENV})')
        ax.text(TURB_X[-1] + 430, 300, f'wind\n{wind_dir_deg:.0f} deg',
                fontsize=9, color='steelblue', ha='center')
        return ax

    _obs0 = env_reset(9001, WIND_OPTS)
    wind_speed_ep0 = float(_obs0['freewind_measurements'][0])
    wind_dir_ep0   = float(_obs0['freewind_measurements'][1])
    yaw_step0 = eval_df[[f'yaw_{i}' for i in range(N_TURBINES)]].iloc[0].values
    yaw_final = eval_df[[f'yaw_{i}' for i in range(N_TURBINES)]].iloc[-1].values

    fig, axes = plt.subplots(2, 1, figsize=(12, 9))
    plot_farm_layout_with_yaw(yaw_step0, wind_speed_ep0, wind_dir_ep0, ax=axes[0], step=0)
    plot_farm_layout_with_yaw(yaw_final, wind_speed_ep0, wind_dir_ep0, ax=axes[1], step=148)
    plt.tight_layout()
    plt.show()
    print(f'Episode 0 wind: {wind_speed_ep0:.2f} m/s from {wind_dir_ep0:.1f} deg')
    print(f'Yaw at step   0: {yaw_step0}')
    print(f'Yaw at step 148: {yaw_final}')


## Animation: Yaw Angles Over an Episode

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

if SELECTED_ENV != "Turb3_Row1_Floris":
    print(f"Animation uses the hardcoded Turb3_Row1 layout; skipping for {SELECTED_ENV}.")
    ani = None
else:
    fig, (ax_farm, ax_yaw) = plt.subplots(1, 2, figsize=(14, 5))
    steps_arr = eval_df['step'].values
    yaw_hist  = np.array([eval_df[f'yaw_{i}'].values for i in range(N_TURBINES)]).T
    c_turb    = sns.color_palette('tab10', N_TURBINES)

    def animate(frame):
        ax_farm.cla()
        ax_yaw.cla()
        plot_farm_layout_with_yaw(yaw_hist[frame], wind_speed_ep0, wind_dir_ep0,
                                  ax=ax_farm, step=frame)
        for i in range(N_TURBINES):
            ax_yaw.plot(steps_arr[:frame+1], yaw_hist[:frame+1, i],
                        color=c_turb[i], label=f'T{i}')
        ax_yaw.set_xlim(0, steps_arr[-1])
        ax_yaw.set_ylim(-50, 50)
        ax_yaw.axhline(0, color='gray', lw=0.8, ls='--')
        ax_yaw.set_xlabel('Step')
        ax_yaw.set_ylabel('Yaw angle (deg)')
        ax_yaw.set_title(f'Yaw angle history ({SELECTED_ENV})')
        ax_yaw.legend(loc='upper left')

    frame_indices = list(range(0, len(steps_arr), 3))
    ani = FuncAnimation(fig, animate, frames=frame_indices, interval=80, blit=False)
    plt.tight_layout()
    plt.close()

HTML(ani.to_jshtml()) if ani is not None else None


## Performance Analysis: When Does PPO Help?

In [ ]:
def evaluate_with_wind_tracking(n_episodes=40, seed_offset=9001):
    zero_action = {'yaw': np.zeros(N_TURBINES, dtype=np.float32)}
    rows = []
    policy.eval()
    with torch.no_grad():
        for ep in range(n_episodes):
            seed = seed_offset + ep

            # PPO rollout
            obs_d = env_reset(seed, WIND_OPTS)
            ws = float(obs_d['freewind_measurements'][0])
            wd = float(obs_d['freewind_measurements'][1])
            done, ppo_r, step = False, 0.0, 0
            while not done:
                obs_t = torch.tensor(flatten_obs(obs_d, step=step), dtype=torch.float32, device=DEVICE).unsqueeze(0)
                action_np = np.clip(
                    policy.actor_mean(policy.actor_backbone(obs_t)).squeeze(0).cpu().numpy(),
                    env.action_space['yaw'].low, env.action_space['yaw'].high,
                )
                obs_d, reward, term, trunc, _ = env.step({'yaw': action_np})
                ppo_r += float(reward[0]) if hasattr(reward, '__len__') else float(reward)
                done = term or trunc
                step += 1

            # Baseline rollout (same seed -> same wind)
            obs_d = env_reset(seed, WIND_OPTS)
            done, base_r = False, 0.0
            while not done:
                obs_d, reward, term, trunc, _ = env.step(zero_action)
                base_r += float(reward[0]) if hasattr(reward, '__len__') else float(reward)
                done = term or trunc

            rows.append({
                'ep': ep, 'wind_speed': ws, 'wind_dir': wd,
                'ppo': ppo_r, 'baseline': base_r,
                'gain': ppo_r - base_r,
                'gain_pct': (ppo_r - base_r) / abs(base_r) * 100,
            })
    policy.train()
    return pd.DataFrame(rows)

perf_df = evaluate_with_wind_tracking(n_episodes=180)
print(perf_df[['wind_speed','wind_dir','ppo','baseline','gain_pct']].round(2).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
sc = ax.scatter(perf_df['wind_dir'], perf_df['gain_pct'],
                c=perf_df['wind_speed'], cmap='viridis', s=60, alpha=0.85)
ax.axhline(0, color='gray', lw=1, ls='--')
ax.set_xlabel('Wind direction (deg FROM)')
ax.set_ylabel('PPO gain over baseline (%)')
ax.set_title('PPO gain vs wind direction\n(color = wind speed)')
plt.colorbar(sc, ax=ax, label='Wind speed (m/s)')

ax = axes[1]
sc = ax.scatter(perf_df['wind_speed'], perf_df['gain_pct'],
                c=perf_df['wind_dir'], cmap='plasma', s=60, alpha=0.85)
ax.axhline(0, color='gray', lw=1, ls='--')
ax.set_xlabel('Wind speed (m/s)')
ax.set_ylabel('PPO gain over baseline (%)')
ax.set_title('PPO gain vs wind speed\n(color = wind direction)')
plt.colorbar(sc, ax=ax, label='Wind direction (deg)')

ax = axes[2]
sorted_df = perf_df.sort_values('baseline').reset_index(drop=True)
bar_colors = ['steelblue' if g >= 0 else 'tomato' for g in sorted_df['gain_pct']]
ax.bar(range(len(sorted_df)), sorted_df['gain_pct'], color=bar_colors)
ax.axhline(0, color='gray', lw=1, ls='--')
ax.set_xlabel('Episode sorted by baseline reward ->')
ax.set_ylabel('PPO gain (%)')
ax.set_title('Gain sorted by episode difficulty\n(blue=PPO better, red=PPO worse)')

plt.suptitle('When Does PPO Help? (40 eval episodes)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'Overall: PPO better in {(perf_df.gain_pct > 0).sum()}/{len(perf_df)} episodes')
print(f'Mean gain: {perf_df.gain_pct.mean():+.2f}%  |  Std: {perf_df.gain_pct.std():.2f}%')
print(f'Wind dir range: {perf_df.wind_dir.min():.0f} - {perf_df.wind_dir.max():.0f} deg')
print(f'Corr(gain_pct, wind_dir):   {perf_df[["gain_pct","wind_dir"]].corr().iloc[0,1]:.3f}')
print(f'Corr(gain_pct, wind_speed): {perf_df[["gain_pct","wind_speed"]].corr().iloc[0,1]:.3f}')
print(f'Corr(gain_pct, baseline):   {perf_df[["gain_pct","baseline"]].corr().iloc[0,1]:.3f}')


## Cross-Environment Summary

Compare PPO performance across all 19 trained environments.

In [ ]:
import pandas as pd

# Build a tidy results DataFrame
rows = []
for env_id, r in results.items():
    if "error" in r:
        rows.append({"env_id": env_id, "status": "FAILED", "error": r["error"]})
    else:
        rows.append({
            "env_id":        env_id,
            "status":        "ok",
            "n_turbines":    r["n_turbines"],
            "ppo_mean":      r["eval_mean"],
            "ppo_std":       r["eval_std"],
            "baseline_mean": r["baseline_mean"],
            "baseline_std":  r["baseline_std"],
            "gain_pct":      r["gain_pct"],
            "simulator":     "Fastfarm" if "Fastfarm" in env_id else "Floris",
        })

results_df = pd.DataFrame(rows)
results_df_ok = results_df[results_df["status"] == "ok"].sort_values("gain_pct", ascending=False).reset_index(drop=True)
print(f"{(results_df['status'] == 'ok').sum()}/{len(ENV_IDS)} envs trained successfully")
print()
results_df


In [ ]:
# Grouped bar chart: PPO vs. no-control baseline per env (sorted by gain%)
ok = results_df_ok
if len(ok) == 0:
    print("No successful runs to plot.")
else:
    x = np.arange(len(ok))
    w = 0.4

    fig, ax = plt.subplots(figsize=(max(10, 0.7 * len(ok)), 6))
    ax.bar(x - w/2, ok["ppo_mean"],      w, yerr=ok["ppo_std"],
           color="#2ca02c", capsize=4, label="PPO (mean +- std)")
    ax.bar(x + w/2, ok["baseline_mean"], w, yerr=ok["baseline_std"],
           color="#7f7f7f", capsize=4, label="No-control baseline")
    ax.set_xticks(x)
    ax.set_xticklabels(ok["env_id"], rotation=45, ha="right")
    ax.set_ylabel("Mean episode reward (Sc.2)")
    ax.set_title("PPO vs. No-Control Baseline across all environments (sorted by gain %)")
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Single-bar gain% chart, color-coded by simulator backend
ok = results_df_ok
if len(ok) > 0:
    fig, ax = plt.subplots(figsize=(max(10, 0.7 * len(ok)), 5))
    bar_colors = ["#1f77b4" if s == "Floris" else "#ff7f0e" for s in ok["simulator"]]
    ax.bar(np.arange(len(ok)), ok["gain_pct"], color=bar_colors, edgecolor="black", lw=0.5)
    ax.axhline(0, color="gray", lw=1, ls="--")
    ax.set_xticks(np.arange(len(ok)))
    ax.set_xticklabels(ok["env_id"], rotation=45, ha="right")
    ax.set_ylabel("PPO gain over baseline (%)")
    ax.set_title("PPO Gain Across Environments  (blue=Floris, orange=Fastfarm)")
    plt.tight_layout()
    plt.show()


In [ ]:
# Scaling plot: gain% vs. turbine count, separated by simulator
ok = results_df_ok
if len(ok) > 0:
    fig, ax = plt.subplots(figsize=(9, 5))
    for sim, color in [("Floris", "#1f77b4"), ("Fastfarm", "#ff7f0e")]:
        sub = ok[ok["simulator"] == sim]
        if len(sub) == 0:
            continue
        ax.scatter(sub["n_turbines"], sub["gain_pct"], s=80, color=color,
                   edgecolor="black", label=sim, alpha=0.85)
        for _, row in sub.iterrows():
            ax.annotate(row["env_id"], (row["n_turbines"], row["gain_pct"]),
                        fontsize=7, xytext=(5, 3), textcoords="offset points")
    ax.axhline(0, color="gray", lw=1, ls="--")
    ax.set_xlabel("Number of turbines")
    ax.set_ylabel("PPO gain over baseline (%)")
    ax.set_title("Does PPO gain scale with farm size?")
    ax.legend()
    plt.tight_layout()
    plt.show()
